# Classification tables for paper

## Setup

In [1]:
# imports
from dataclasses import dataclass
from __future__ import annotations
import numpy as np
import pandas as pd
from pathlib import Path
import re
from typing import Dict, List, Optional, Tuple


In [2]:
# config

OUTDIR = Path("/midtier/paetzollab/scratch/ads4015/lsm_fm_paper/tables")
OUTDIR.mkdir(parents=True, exist_ok=True)

FOLDS = [0, 1, 2]
NTRS = [24, 56, 105]
DECIMALS = 2

ROOTS: Dict[str, Path] = {
    "Unet Image+CLIP": Path("/midtier/paetzollab/scratch/ads4015/temp_selma_classification_preds_super_sweep2"),
    "Unet Image-only": Path("/midtier/paetzollab/scratch/ads4015/temp_selma_classification_preds_bright_sweep_26"),
    "Unet Random-init": Path("/midtier/paetzollab/scratch/ads4015/temp_selma_classification_preds_unet_random2"),
    "SwinUNETR Image+CLIP": Path("/midtier/paetzollab/scratch/ads4015/temp_selma_classification_preds_autumn_sweep_27"),
    "SwinUNETR Image-only": Path("/midtier/paetzollab/scratch/ads4015/temp_selma_classification_preds_expert_sweep_31"),
    "SwinUNETR Random-init": Path("/midtier/paetzollab/scratch/ads4015/temp_selma_classification_preds_random"),
    "PCA+LR": Path("/midtier/paetzollab/scratch/ads4015/temp_selma_classification_preds_pca_lr"),
    "ResNet": Path("/midtier/paetzollab/scratch/ads4015/temp_selma_classification_preds_resnet"),
}

MODEL_ORDER = [
    "Unet Image+CLIP",
    "SwinUNETR Image+CLIP",
    "Unet Image-only",
    "SwinUNETR Image-only",
    "Unet Random-init",
    "SwinUNETR Random-init",
    "PCA+LR",
    "ResNet",
]

# folder + file patterns
BASE_METRICS_DIR_CANDIDATES = [
    Path("cls_metrics") / "classification",
    Path("cls_metrics") / "classification_unet",
]

# naming of run directory
RUN_PATTERNS = [
    # pretrained / random / pca_lr / resnet — we only rely on cvfold + ntr being in the name
    re.compile(r"cvfold(?P<fold>\d+)_ntr(?P<ntr>\d+)(?:_|$)"),
]

# per_class_metrics filename prefixes by model family
PREFER_PREFIX = {
    "Unet Image+CLIP": "per_class_metrics_pretrained_",
    "Unet Image-only": "per_class_metrics_pretrained_",
    "SwinUNETR Image+CLIP": "per_class_metrics_pretrained_",
    "SwinUNETR Image-only": "per_class_metrics_pretrained_",
    "Unet Random-init": "per_class_metrics_random_",
    "SwinUNETR Random-init": "per_class_metrics_random_",
    "PCA+LR": "per_class_metrics_pca_lr_",
    "ResNet": "per_class_metrics_resnet_",
}

# constant for overall row in per-class metrics
OVERALL_ROW = "__OVERALL__"


## Helper functions

In [3]:
# safe mean
def safe_mean(xs: List[float]) -> float:
    xs = [x for x in xs if x is not None and np.isfinite(x)]
    return float(np.mean(xs)) if xs else float("nan")


# find run_dir for model / fold / ntr
# either <model_root>/cls_metrics/classification/ or <model_root>/cls_metrics/classification_unet/
def pick_run_dir(model_root: Path, fold: int, ntr: int) -> Optional[Path]:

    pat = re.compile(rf"cvfold{fold}_ntr{ntr}(?:_|$)")
    run_dirs: List[Path] = []

    for rel in BASE_METRICS_DIR_CANDIDATES:
        base = model_root / rel
        if not base.exists():
            continue
        run_dirs.extend([d for d in base.iterdir() if d.is_dir() and pat.search(d.name)])

    if not run_dirs:
        return None

    run_dirs.sort(key=lambda d: d.stat().st_mtime, reverse=True)
    return run_dirs[0]


# find per_class_metrics*.csv in run_dir, prefering a given prefix if provided
def find_metrics_csv(run_dir: Path, prefer_prefix: Optional[str] = None) -> Optional[Path]:

    if run_dir is None or (not run_dir.exists()):
        return None

    csvs = sorted(run_dir.glob("per_class_metrics*.csv"))
    if not csvs:
        # sometimes people nest one more folder; be robust
        csvs = sorted(run_dir.rglob("per_class_metrics*.csv"))
    if not csvs:
        return None

    if prefer_prefix:
        preferred = [p for p in csvs if p.name.startswith(prefer_prefix)]
        if preferred:
            # if multiple, choose most recent
            preferred.sort(key=lambda p: p.stat().st_mtime, reverse=True)
            return preferred[0]

    # fallback: most recent
    csvs.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return csvs[0]

# parse overall ACC and MACRO_F1 from per_class_metrics CSV
def parse_overall_acc_macro_f1(csv_path: Path) -> Tuple[float, float]:

    dfm = pd.read_csv(csv_path)
    overall = dfm[dfm["class_name"] == OVERALL_ROW]
    if overall.empty:
        raise ValueError(f"No {OVERALL_ROW} row found in {csv_path}")

    row = overall.iloc[0].tolist()

    # robust parse: scan all cells and look for ACC= and MACRO_F1=
    acc = None
    macro_f1 = None
    for cell in row:
        if isinstance(cell, str):
            m1 = re.search(r"ACC\s*=\s*([0-9]*\.?[0-9]+)", cell)
            if m1:
                acc = float(m1.group(1))
            m2 = re.search(r"MACRO_F1\s*=\s*([0-9]*\.?[0-9]+)", cell)
            if m2:
                macro_f1 = float(m2.group(1))

    if acc is None or macro_f1 is None:
        raise ValueError(f"Could not parse ACC/MACRO_F1 from {csv_path}")

    return float(acc), float(macro_f1)


# evaluate model + ntr across folds — average ACC + MACRO_F1 across folds for a given model + ntr
def eval_model_ntr(model_name: str, root: Path, ntr: int) -> Dict[str, float]:
    """
    Average ACC + MACRO_F1 across folds for a given model + ntr.
    """
    accs, f1s = [], []
    for fold in FOLDS:
        run_dir = pick_run_dir(root, fold=fold, ntr=ntr)
        if run_dir is None:
            accs.append(np.nan); f1s.append(np.nan)
            continue

        csv_path = find_metrics_csv(run_dir, prefer_prefix=PREFER_PREFIX.get(model_name))
        if csv_path is None:
            accs.append(np.nan); f1s.append(np.nan)
            continue

        try:
            acc, f1 = parse_overall_acc_macro_f1(csv_path)
        except Exception:
            acc, f1 = np.nan, np.nan

        accs.append(acc)
        f1s.append(f1)

    return {"ACC": safe_mean(accs), "MACRO_F1": safe_mean(f1s)}


# bold best per column (LaTeX strings)
def latex_bold_max_per_column(df_num: pd.DataFrame, decimals: int = 2, na_rep: str = "—") -> pd.DataFrame:
    fmt = f"{{:.{decimals}f}}"
    out = pd.DataFrame(index=df_num.index, columns=df_num.columns, dtype=object)

    for col in df_num.columns:
        s = pd.to_numeric(df_num[col], errors="coerce")
        maxv = s.max(skipna=True)
        col_out = []
        for v in s.values:
            if not np.isfinite(v):
                col_out.append(na_rep)
            else:
                txt = fmt.format(float(v))
                if np.isfinite(maxv) and np.isclose(v, maxv, rtol=0, atol=1e-12):
                    txt = rf"\textbf{{{txt}}}"
                col_out.append(txt)
        out[col] = col_out
    return out


# specify LaTeX tabular colspec for n numeric columns
def latex_tabular_colspec(n_numeric_cols: int) -> str:
    return "@{}" + "l" + ("c" * n_numeric_cols) + "@{}"


# replace first tabular colspec in LaTeX string
def replace_first_tabular_colspec(latex_str: str, colspec: str) -> str:
    return re.sub(
        r"\\begin\{tabular\}\{[^}]*\}",
        rf"\\begin{{tabular}}{{{colspec}}}",
        latex_str,
        count=1,
    )


# inject formatting commands into LaTeX table string
def inject_table_formatting(
    latex_str: str,
    add_centering: bool = True,
    fontsize_cmd: str = r"\fontsize{8}{9}\selectfont",
    tabcolsep_pt: int = 2,
    arraystretch: float = 1.05,
) -> str:
    lines = latex_str.splitlines()
    out = []
    injected = False
    for line in lines:
        out.append(line)
        s = line.strip()
        if (not injected) and (s.startswith(r"\begin{table}") or s.startswith(r"\begin{sidewaystable}")):
            if add_centering:
                out.append(r"\centering")
            out.append(fontsize_cmd)
            out.append(rf"\setlength{{\tabcolsep}}{{{tabcolsep_pt}pt}}")
            out.append(rf"\renewcommand{{\arraystretch}}{{{arraystretch}}}")
            injected = True
    return "\n".join(out)


# shrink first column with your abbreviations
def _latex_wrap_model_name(name: str) -> str:
    mapping = {
        "Unet Image+CLIP": r"\makecell[l]{UNet Img+Txt}",
        "Unet Image-only": r"\makecell[l]{UNet Img}",
        "Unet Random-init": r"\makecell[l]{UNet Rand}",
        "SwinUNETR Image+CLIP": r"\makecell[l]{SwinUNETR Img+Txt}",
        "SwinUNETR Image-only": r"\makecell[l]{SwinUNETR Img}",
        "SwinUNETR Random-init": r"\makecell[l]{SwinUNETR Rand}",
        "PCA+LR": r"\makecell[l]{PCA+LR}",
        "ResNet": r"\makecell[l]{ResNet}",
    }
    return mapping.get(name, name)


In [4]:
# build table

# build DataFrame with results
records = []
for model in MODEL_ORDER:
    root = ROOTS[model]
    row = {"Model": model}
    for ntr in NTRS:
        m = eval_model_ntr(model, root, ntr=ntr)
        row[(f"samples={ntr}", "ACC")] = m["ACC"]
        row[(f"samples={ntr}", r"MACRO $F_1$")] = m["MACRO_F1"]
    records.append(row)

df_cls = pd.DataFrame.from_records(records).set_index("Model")
df_cls.columns = pd.MultiIndex.from_tuples(df_cls.columns)

# save csv
csv_out = OUTDIR / "classification_results.csv"
df_cls.round(6).to_csv(csv_out)
print(f"[Saved] {csv_out}")

# latex export with formatting
df_tex = latex_bold_max_per_column(df_cls.round(DECIMALS), decimals=DECIMALS, na_rep="—")
df_tex.index = [_latex_wrap_model_name(str(i)) for i in df_tex.index]

latex_str = df_tex.to_latex(
    escape=False,
    multicolumn=True,
    multirow=True,
    caption=(
        "Classification performance (Accuracy and Macro-F1) averaged across 3 CV folds "
        "for training sizes ntr=24, ntr=56, and ntr=105."
    ),
    label="tab:classification_results",
    bold_rows=False,
    longtable=False,
    index=True,
)

# Center ntr headers across ACC/MACRO_F1
latex_str = latex_str.replace(r"\multicolumn{2}{r}{samples=24}",  r"\multicolumn{2}{c}{samples=24}")
latex_str = latex_str.replace(r"\multicolumn{2}{r}{samples=56}",  r"\multicolumn{2}{c}{samples=56}")
latex_str = latex_str.replace(r"\multicolumn{2}{r}{samples=105}", r"\multicolumn{2}{c}{samples=105}")

# tight colspec: Model + (3 ntr blocks * 2 metrics) = 6 numeric cols
latex_str = replace_first_tabular_colspec(latex_str, latex_tabular_colspec(n_numeric_cols=6))

# inject formatting (for compactness)
latex_str = inject_table_formatting(
    latex_str,
    add_centering=True,
    fontsize_cmd=r"\fontsize{8}{9}\selectfont",
    tabcolsep_pt=3,
    arraystretch=1.1,
)

# save tex
tex_out = OUTDIR / "classification_results.tex"
tex_out.write_text(latex_str)
print(f"[Saved] {tex_out}")

# display in notebook
display(df_cls.round(DECIMALS))


[Saved] /midtier/paetzollab/scratch/ads4015/lsm_fm_paper/tables/classification_results.csv
[Saved] /midtier/paetzollab/scratch/ads4015/lsm_fm_paper/tables/classification_results.tex


samples=24             samples=56              \
                             ACC MACRO $F_1$        ACC MACRO $F_1$   
Model                                                                 
Unet Image+CLIP             0.38        0.34       0.51        0.47   
SwinUNETR Image+CLIP        0.33        0.27       0.40        0.35   
Unet Image-only             0.36        0.33       0.56        0.51   
SwinUNETR Image-only        0.38        0.31       0.51        0.46   
Unet Random-init            0.44        0.40       0.54        0.51   
SwinUNETR Random-init       0.25        0.17       0.40        0.33   
PCA+LR                      0.33        0.26       0.33        0.28   
ResNet                      0.28        0.22       0.36        0.27   

                      samples=105              
                              ACC MACRO $F_1$  
Model                                          
Unet Image+CLIP              0.71        0.67  
SwinUNETR Image+CLIP         0.65        0.61  
Unet Image-only              0.76        0.74  
SwinUNETR Image-only         0.53        0.48  
Unet Random-init             0.58        0.53  
SwinUNETR Random-init        0.47        0.44  
PCA+LR                       0.47        0.40  
ResNet                       0.49        0.43

In [5]:
# ============================================================
# CLASSIFICATION RESULTS TABLE (ACC + MACRO_F1)
# Averaged across cvfold0/1/2 for ntr in {24, 56, 105}
# Exports CSV + LaTeX (MICCAI-ready)
# ============================================================

from __future__ import annotations
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple
import re
import numpy as np
import pandas as pd

# ---------------------------
# CONFIG
# ---------------------------

OUTDIR = Path("/midtier/paetzollab/scratch/ads4015/lsm_fm_paper/tables")
OUTDIR.mkdir(parents=True, exist_ok=True)

FOLDS = [0, 1, 2]
NTRS = [24, 56, 105]
DECIMALS = 2

ROOTS: Dict[str, Path] = {
    "Unet Image+CLIP": Path("/midtier/paetzollab/scratch/ads4015/temp_selma_classification_preds_super_sweep2"),
    "Unet Image-only": Path("/midtier/paetzollab/scratch/ads4015/temp_selma_classification_preds_bright_sweep_26"),
    "Unet Random-init": Path("/midtier/paetzollab/scratch/ads4015/temp_selma_classification_preds_unet_random2"),
    "SwinUNETR Image+CLIP": Path("/midtier/paetzollab/scratch/ads4015/temp_selma_classification_preds_autumn_sweep_27"),
    "SwinUNETR Image-only": Path("/midtier/paetzollab/scratch/ads4015/temp_selma_classification_preds_expert_sweep_31"),
    "SwinUNETR Random-init": Path("/midtier/paetzollab/scratch/ads4015/temp_selma_classification_preds_random"),
    "PCA+LR": Path("/midtier/paetzollab/scratch/ads4015/temp_selma_classification_preds_pca_lr"),
    "ResNet": Path("/midtier/paetzollab/scratch/ads4015/temp_selma_classification_preds_resnet"),
}

MODEL_ORDER = [
    "Unet Image+CLIP",
    "SwinUNETR Image+CLIP",
    "Unet Image-only",
    "SwinUNETR Image-only",
    "Unet Random-init",
    "SwinUNETR Random-init",
    "PCA+LR",
    "ResNet",
]

# Folder + file patterns
BASE_METRICS_DIR_CANDIDATES = [
    Path("cls_metrics") / "classification",
    Path("cls_metrics") / "classification_unet",
]

# run dir naming
RUN_PATTERNS = [
    # pretrained / random / pca_lr / resnet — we only rely on cvfold + ntr being in the name
    re.compile(r"cvfold(?P<fold>\d+)_ntr(?P<ntr>\d+)(?:_|$)"),
]

# per_class_metrics filename prefixes by model family
# (we'll search for any per_class_metrics*.csv inside the chosen run_dir, but this helps prefer correct file)
PREFER_PREFIX = {
    "Unet Image+CLIP": "per_class_metrics_pretrained_",
    "Unet Image-only": "per_class_metrics_pretrained_",
    "SwinUNETR Image+CLIP": "per_class_metrics_pretrained_",
    "SwinUNETR Image-only": "per_class_metrics_pretrained_",
    "Unet Random-init": "per_class_metrics_random_",
    "SwinUNETR Random-init": "per_class_metrics_random_",
    "PCA+LR": "per_class_metrics_pca_lr_",
    "ResNet": "per_class_metrics_resnet_",
}

# ---------------------------
# HELPERS
# ---------------------------

def safe_mean(xs: List[float]) -> float:
    xs = [x for x in xs if x is not None and np.isfinite(x)]
    return float(np.mean(xs)) if xs else float("nan")

def pick_run_dir(model_root: Path, fold: int, ntr: int) -> Optional[Path]:
    """
    Finds the most recent run dir under either:
      <model_root>/cls_metrics/classification/
      <model_root>/cls_metrics/classification_unet/
    """
    pat = re.compile(rf"cvfold{fold}_ntr{ntr}(?:_|$)")
    run_dirs: List[Path] = []

    for rel in BASE_METRICS_DIR_CANDIDATES:
        base = model_root / rel
        if not base.exists():
            continue
        run_dirs.extend([d for d in base.iterdir() if d.is_dir() and pat.search(d.name)])

    if not run_dirs:
        return None

    run_dirs.sort(key=lambda d: d.stat().st_mtime, reverse=True)
    return run_dirs[0]


def find_metrics_csv(run_dir: Path, prefer_prefix: Optional[str] = None) -> Optional[Path]:
    """
    Returns the best-matching per_class_metrics*.csv in the run_dir.
    Prefers a given prefix if provided.
    """
    if run_dir is None or (not run_dir.exists()):
        return None

    csvs = sorted(run_dir.glob("per_class_metrics*.csv"))
    if not csvs:
        # sometimes people nest one more folder; be robust
        csvs = sorted(run_dir.rglob("per_class_metrics*.csv"))
    if not csvs:
        return None

    if prefer_prefix:
        preferred = [p for p in csvs if p.name.startswith(prefer_prefix)]
        if preferred:
            # if multiple, choose most recent
            preferred.sort(key=lambda p: p.stat().st_mtime, reverse=True)
            return preferred[0]

    # fallback: most recent
    csvs.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return csvs[0]

OVERALL_ROW = "__OVERALL__"

def parse_overall_acc_macro_f1(csv_path: Path) -> Tuple[float, float]:
    """
    Parses the __OVERALL__ line like:
      __OVERALL__,24,ACC=0.208333,MACRO_F1=0.138889,
    Returns (acc, macro_f1).
    """
    dfm = pd.read_csv(csv_path)
    overall = dfm[dfm["class_name"] == OVERALL_ROW]
    if overall.empty:
        raise ValueError(f"No {OVERALL_ROW} row found in {csv_path}")

    row = overall.iloc[0].tolist()

    # robust parse: scan all cells and look for ACC= and MACRO_F1=
    acc = None
    macro_f1 = None
    for cell in row:
        if isinstance(cell, str):
            m1 = re.search(r"ACC\s*=\s*([0-9]*\.?[0-9]+)", cell)
            if m1:
                acc = float(m1.group(1))
            m2 = re.search(r"MACRO_F1\s*=\s*([0-9]*\.?[0-9]+)", cell)
            if m2:
                macro_f1 = float(m2.group(1))

    if acc is None or macro_f1 is None:
        raise ValueError(f"Could not parse ACC/MACRO_F1 from {csv_path}")

    return float(acc), float(macro_f1)

def eval_model_ntr(model_name: str, root: Path, ntr: int) -> Dict[str, float]:
    """
    Average ACC + MACRO_F1 across folds for a given model + ntr.
    """
    accs, f1s = [], []
    for fold in FOLDS:
        run_dir = pick_run_dir(root, fold=fold, ntr=ntr)
        if run_dir is None:
            accs.append(np.nan); f1s.append(np.nan)
            continue

        csv_path = find_metrics_csv(run_dir, prefer_prefix=PREFER_PREFIX.get(model_name))
        if csv_path is None:
            accs.append(np.nan); f1s.append(np.nan)
            continue

        try:
            acc, f1 = parse_overall_acc_macro_f1(csv_path)
        except Exception:
            acc, f1 = np.nan, np.nan

        accs.append(acc)
        f1s.append(f1)

    return {"ACC": safe_mean(accs), "MACRO_F1": safe_mean(f1s)}

# Bold best per column (LaTeX strings)
def latex_bold_max_per_column(df_num: pd.DataFrame, decimals: int = 2, na_rep: str = "—") -> pd.DataFrame:
    fmt = f"{{:.{decimals}f}}"
    out = pd.DataFrame(index=df_num.index, columns=df_num.columns, dtype=object)

    for col in df_num.columns:
        s = pd.to_numeric(df_num[col], errors="coerce")
        maxv = s.max(skipna=True)
        col_out = []
        for v in s.values:
            if not np.isfinite(v):
                col_out.append(na_rep)
            else:
                txt = fmt.format(float(v))
                if np.isfinite(maxv) and np.isclose(v, maxv, rtol=0, atol=1e-12):
                    txt = rf"\textbf{{{txt}}}"
                col_out.append(txt)
        out[col] = col_out
    return out

def latex_tabular_colspec(n_numeric_cols: int) -> str:
    return "@{}" + "l" + ("c" * n_numeric_cols) + "@{}"

def replace_first_tabular_colspec(latex_str: str, colspec: str) -> str:
    return re.sub(
        r"\\begin\{tabular\}\{[^}]*\}",
        rf"\\begin{{tabular}}{{{colspec}}}",
        latex_str,
        count=1,
    )

# If you already have this from your segmentation script, reuse it.
def inject_table_formatting(
    latex_str: str,
    add_centering: bool = True,
    fontsize_cmd: str = r"\fontsize{8}{9}\selectfont",
    tabcolsep_pt: int = 2,
    arraystretch: float = 1.05,
) -> str:
    lines = latex_str.splitlines()
    out = []
    injected = False
    for line in lines:
        out.append(line)
        s = line.strip()
        if (not injected) and (s.startswith(r"\begin{table}") or s.startswith(r"\begin{sidewaystable}")):
            if add_centering:
                out.append(r"\centering")
            out.append(fontsize_cmd)
            out.append(rf"\setlength{{\tabcolsep}}{{{tabcolsep_pt}pt}}")
            out.append(rf"\renewcommand{{\arraystretch}}{{{arraystretch}}}")
            injected = True
    return "\n".join(out)

# Optional: shrink first column with your abbreviations
def _latex_wrap_model_name(name: str) -> str:
    mapping = {
        "Unet Image+CLIP": r"\makecell[l]{UNet Img+Txt}",
        "Unet Image-only": r"\makecell[l]{UNet Img}",
        "Unet Random-init": r"\makecell[l]{UNet Rand}",
        "SwinUNETR Image+CLIP": r"\makecell[l]{SwinUNETR Img+Txt}",
        "SwinUNETR Image-only": r"\makecell[l]{SwinUNETR Img}",
        "SwinUNETR Random-init": r"\makecell[l]{SwinUNETR Rand}",
        "PCA+LR": r"\makecell[l]{PCA+LR}",
        "ResNet": r"\makecell[l]{ResNet}",
    }
    return mapping.get(name, name)

# ---------------------------
# BUILD TABLE
# ---------------------------

records = []
for model in MODEL_ORDER:
    root = ROOTS[model]
    row = {"Model": model}
    for ntr in NTRS:
        m = eval_model_ntr(model, root, ntr=ntr)
        row[(f"samples={ntr}", "ACC")] = m["ACC"]
        row[(f"samples={ntr}", r"MACRO $F_1$")] = m["MACRO_F1"]
    records.append(row)

df_cls = pd.DataFrame.from_records(records).set_index("Model")
df_cls.columns = pd.MultiIndex.from_tuples(df_cls.columns)

# Save numeric CSV
csv_out = OUTDIR / "classification_results.csv"
df_cls.round(6).to_csv(csv_out)
print(f"[Saved] {csv_out}")

# ---------------------------
# LaTeX export (bold best per column)
# ---------------------------

df_tex = latex_bold_max_per_column(df_cls.round(DECIMALS), decimals=DECIMALS, na_rep="—")
df_tex.index = [_latex_wrap_model_name(str(i)) for i in df_tex.index]

latex_str = df_tex.to_latex(
    escape=False,
    multicolumn=True,
    multirow=True,
    caption=(
        "Classification performance (Accuracy and Macro-F1) averaged across 3 CV folds "
        "for training sizes ntr=24, ntr=56, and ntr=105."
    ),
    label="tab:classification_results",
    bold_rows=False,
    longtable=False,
    index=True,
)

# Center ntr headers across ACC/MACRO_F1 (pandas sometimes emits {r})
latex_str = latex_str.replace(r"\multicolumn{2}{r}{samples=24}",  r"\multicolumn{2}{c}{samples=24}")
latex_str = latex_str.replace(r"\multicolumn{2}{r}{samples=56}",  r"\multicolumn{2}{c}{samples=56}")
latex_str = latex_str.replace(r"\multicolumn{2}{r}{samples=105}", r"\multicolumn{2}{c}{samples=105}")

# Tight colspec: Model + (3 ntr blocks * 2 metrics) = 6 numeric cols
latex_str = replace_first_tabular_colspec(latex_str, latex_tabular_colspec(n_numeric_cols=6))

# Inject formatting for MICCAI compactness
latex_str = inject_table_formatting(
    latex_str,
    add_centering=True,
    fontsize_cmd=r"\fontsize{8}{9}\selectfont",
    tabcolsep_pt=3,
    arraystretch=1.1,
)

tex_out = OUTDIR / "classification_results.tex"
tex_out.write_text(latex_str)
print(f"[Saved] {tex_out}")

# Optional display in notebook
display(df_cls.round(DECIMALS))


[Saved] /midtier/paetzollab/scratch/ads4015/lsm_fm_paper/tables/classification_results.csv
[Saved] /midtier/paetzollab/scratch/ads4015/lsm_fm_paper/tables/classification_results.tex


samples=24             samples=56              \
                             ACC MACRO $F_1$        ACC MACRO $F_1$   
Model                                                                 
Unet Image+CLIP             0.38        0.34       0.51        0.47   
SwinUNETR Image+CLIP        0.33        0.27       0.40        0.35   
Unet Image-only             0.36        0.33       0.56        0.51   
SwinUNETR Image-only        0.38        0.31       0.51        0.46   
Unet Random-init            0.44        0.40       0.54        0.51   
SwinUNETR Random-init       0.25        0.17       0.40        0.33   
PCA+LR                      0.33        0.26       0.33        0.28   
ResNet                      0.28        0.22       0.36        0.27   

                      samples=105              
                              ACC MACRO $F_1$  
Model                                          
Unet Image+CLIP              0.71        0.67  
SwinUNETR Image+CLIP         0.65        0.61  
Unet Image-only              0.76        0.74  
SwinUNETR Image-only         0.53        0.48  
Unet Random-init             0.58        0.53  
SwinUNETR Random-init        0.47        0.44  
PCA+LR                       0.47        0.40  
ResNet                       0.49        0.43